In [2]:
import pandas as pd  
import numpy as np  
import os  
import re  
import glob  
import io
import calendar
from datetime import datetime  
import msoffcrypto
import warnings
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [3]:
# ============================================================  
# CONFIG  
# ============================================================  
base_paths = [  
    'L:/2025 Pilar Plant Files/2025 Manpower Detail/',  
    'L:/2026 Pilar Plant Files/2026 Manpower Detail/',  
]  
password     = 'bamhorse2'  
SITE_KEYWORD = 'LENOX SAL'          # only files whose name contains this

# Semi-monthly: 15th or last-of-month, Jan 1 2025 -> Mar 31 2026  
SEMI_START = pd.Timestamp('2025-01-01')  
SEMI_END   = pd.Timestamp('2026-03-31')

# Biweekly: everything NOT on the 15th/last-of-month, Mar 1 2026 -> today  
BIWK_START = pd.Timestamp('2026-03-01')  
BIWK_END   = pd.Timestamp.today().normalize()      # 2026-09-25

# Dates that ARE a 15th/EOM but should still be treated as biweekly  
BIWK_EXCEPTIONS = {pd.Timestamp('2026-05-31')}

SHEET_NAME = 'PPE LDR'  
SKIPROWS   = 6          # <-- VERIFY: matched to 'MTD LDR' from the original script  
# ============================================================


def get_workbook_stream(file_path, password):  
    """  
    Return (BytesIO, was_encrypted) for an .xlsm that may or may not be  
    password-protected.  
      - unencrypted OOXML  -> ZIP container, starts with b'PK'  
      - encrypted OOXML    -> OLE container, starts with b'\\xd0\\xcf\\x11\\xe0'  
    """  
    with open(file_path, 'rb') as f:  
        raw = io.BytesIO(f.read())

    raw.seek(0)  
    magic = raw.read(8)  
    raw.seek(0)

    if magic[:2] == b'PK':                 # plain zip -> not encrypted  
        return raw, False

    if magic[:4] == b'\xd0\xcf\x11\xe0':   # OLE -> encrypted  
        office_file = msoffcrypto.OfficeFile(raw)  
        if not office_file.is_encrypted():  
            raw.seek(0)  
            return raw, False  
        office_file.load_key(password=password)  
        decrypted = io.BytesIO()  
        office_file.decrypt(decrypted)  
        decrypted.seek(0)  
        return decrypted, True

    raise ValueError(f"Unrecognized file signature: {magic!r}")


def read_protected_excel(file_path, password, sheet_name, skiprows):  
    """Read a single sheet from an .xlsm, encrypted or not."""  
    stream, was_encrypted = get_workbook_stream(file_path, password)  
    stream.seek(0)  
    df = pd.read_excel(stream, sheet_name=sheet_name, skiprows=skiprows)  
    return df, was_encrypted


def peek_sheet(file_path, password, sheet_name=None, nrows=15):  
    """Dump the top rows (no skiprows) to confirm the tab name / header offset."""  
    stream, _ = get_workbook_stream(file_path, password)  
    stream.seek(0)  
    if sheet_name is None:  
        print(pd.ExcelFile(stream).sheet_names)  
        return None  
    return pd.read_excel(stream, sheet_name=sheet_name, header=None, nrows=nrows)


def is_period_end(d):  
    """True if the date is the 15th or the last calendar day of the month."""  
    last_day = calendar.monthrange(d.year, d.month)[1]  
    return d.day == 15 or d.day == last_day


def classify(d):  
    """Return 'Semi-Monthly', 'Biweekly', or None (skip) for a file date."""  
    # biweekly first so the exception dates win  
    if BIWK_START <= d <= BIWK_END and (d in BIWK_EXCEPTIONS or not is_period_end(d)):  
        return 'Biweekly'  
    if SEMI_START <= d <= SEMI_END and is_period_end(d):  
        return 'Semi-Monthly'  
    return None


# ------------------------------------------------------------  
# Find + classify files across all folders  
# ------------------------------------------------------------  
DATE_RE = re.compile(r'(\d{4}-\d{2}-\d{2})')

all_files = []  
for bp in base_paths:  
    if not os.path.isdir(bp):  
        print(f"WARNING: folder not found -> {bp}")  
        continue  
    found = glob.glob(os.path.join(bp, '*.xlsm'))  
    print(f"{len(found):>4} .xlsm file(s) in {bp}")  
    all_files.extend(found)

files, skipped, no_date = [], [], []  
seen = set()

for fp in all_files:  
    fn = os.path.basename(fp)

    if SITE_KEYWORD.upper() not in fn.upper():  
        continue

    m = DATE_RE.search(fn)  
    if not m:  
        no_date.append(fn)  
        continue

    d = pd.Timestamp(m.group(1))  
    pay_type = classify(d)

    if pay_type is None:  
        skipped.append((fn, d))  
        continue

    # guard against the same report existing in both folders  
    key = fn.lower()  
    if key in seen:  
        print(f"  Duplicate filename skipped: {fp}")  
        continue  
    seen.add(key)

    files.append((fp, pay_type, d))

files.sort(key=lambda x: (x[2], x[0]))   # chronological

print(f"\nMatched {len(files)} file(s) containing '{SITE_KEYWORD}'\n")  
for pt in ['Semi-Monthly', 'Biweekly']:  
    subset = [f for f in files if f[1] == pt]  
    print(f"{pt} ({len(subset)}):")  
    for fp, _, d in subset:  
        print(f"  {d.date()}  {os.path.basename(fp)}")  
    print()

if skipped:  
    print(f"Skipped {len(skipped)} LENOX file(s) outside the date rules:")  
    for fn, d in sorted(skipped, key=lambda x: x[1]):  
        print(f"  {d.date()}  {fn}")  
if no_date:  
    print(f"\n{len(no_date)} LENOX file(s) had no YYYY-MM-DD in the name:")  
    for fn in no_date:  
        print(f"  {fn}")


# ------------------------------------------------------------  
# Read PPE LDR  
# ------------------------------------------------------------  
all_ppe_ldr, errors, unencrypted = [], [], []

for file_path, pay_type, date in files:  
    filename = os.path.basename(file_path)  
    site = filename.split(' ')[0].strip()

    try:  
        df, was_encrypted = read_protected_excel(file_path, password, SHEET_NAME, SKIPROWS)

        tag = '' if was_encrypted else '  [NOT ENCRYPTED]'  
        print(f"Reading {pay_type:<12} | Site: {site:<10} | Date: {date.date()}{tag}")  
        if not was_encrypted:  
            unencrypted.append((filename, date))

        df['Pay_Type']    = pay_type  
        df['Report_Date'] = date  
        df['BU']          = site  
        df['Source_File'] = filename  
        all_ppe_ldr.append(df)

    except Exception as e:  
        print(f"Reading {pay_type:<12} | Site: {site:<10} | Date: {date.date()}")  
        print(f"  ERROR: {type(e).__name__}: {e}")  
        errors.append((filename, f"{type(e).__name__}: {e}"))


# ------------------------------------------------------------  
# Combine + clean column names  
# ------------------------------------------------------------  
ppe_ldr_combined = pd.concat(all_ppe_ldr, ignore_index=True) if all_ppe_ldr else pd.DataFrame()

if not ppe_ldr_combined.empty:  
    ppe_ldr_combined.columns = (ppe_ldr_combined.columns.astype(str).str.strip()  
                                .str.replace(r'[^a-zA-Z0-9]', '_', regex=True)  
                                .str.replace(r'_+', '_', regex=True)  
                                .str.strip('_').str.lower())

print(f"\n--- Summary ---")  
print(f"Files read : {len(files) - len(errors)} of {len(files)}")  
print(f"PPE LDR    : {ppe_ldr_combined.shape}")  
if not ppe_ldr_combined.empty:  
    print(f"Sites found: {ppe_ldr_combined['bu'].unique()}")  
    print(f"Date range : {ppe_ldr_combined['report_date'].min().date()} -> {ppe_ldr_combined['report_date'].max().date()}")  
    print(ppe_ldr_combined.groupby('pay_type')['report_date'].nunique().rename('n_dates'))  
if unencrypted:  
    print(f"\n{len(unencrypted)} file(s) were NOT encrypted:")  
    for fn, d in unencrypted:  
        print(f"  {d.date()}  {fn}")  
if errors:  
    print(f"\n{len(errors)} file(s) failed:")  
    for fn, msg in errors:  
        print(f"  {fn}: {msg}")  

 147 .xlsm file(s) in L:/2025 Pilar Plant Files/2025 Manpower Detail/
 140 .xlsm file(s) in L:/2026 Pilar Plant Files/2026 Manpower Detail/

Matched 45 file(s) containing 'LENOX SAL'

Semi-Monthly (30):
  2025-01-15  LENOX SALAERN LENOX_EXEC 2025-01-15.xlsm
  2025-01-31  LENOX SALAERN LENOX_EXEC 2025-01-31.xlsm
  2025-02-15  LENOX SALAERN LENOX_EXEC 2025-02-15.xlsm
  2025-02-28  LENOX SALAERN LENOX_EXEC 2025-02-28.xlsm
  2025-03-15  LENOX SALAERN LENOX_EXEC 2025-03-15.xlsm
  2025-03-31  LENOX SALAERN LENOX_EXEC 2025-03-31.xlsm
  2025-04-15  LENOX SALAERN LENOX_EXEC 2025-04-15.xlsm
  2025-04-30  LENOX SALAERN LENOX_EXEC 2025-04-30.xlsm
  2025-05-15  LENOX SALAERN LENOX_EXEC 2025-05-15.xlsm
  2025-05-31  LENOX SALAERN LENOX_EXEC 2025-05-31.xlsm
  2025-06-15  LENOX SALAERN LENOX_EXEC 2025-06-15.xlsm
  2025-06-30  LENOX SALAERN LENOX_EXEC 2025-06-30.xlsm
  2025-07-15  LENOX SALAERN LENOX_EXEC 2025-07-15.xlsm
  2025-07-31  LENOX SALAERN LENOX_EXEC 2025-07-31.xlsm
  2025-08-15  LENOX SALAERN

In [4]:
folder_path = 'L:/2026 Pilar Plant Files/Position Control/'

all_excel_files = glob.glob(os.path.join(folder_path, '*.xlsx'))

lenox_files = [  
    f for f in all_excel_files   
    if 'LENOX' in os.path.basename(f).upper()   
    and not os.path.basename(f).startswith('~$')  
]

all_data = {}

for file in lenox_files:  
    file_name = os.path.basename(file)  
    try:  
        df = pd.read_excel(file, sheet_name='PositionDetail', skiprows=12)  
        df['SourceFile'] = file_name  
        all_data[file_name] = df  
        print(f"Loaded: {file_name} ({len(df)} rows)")  
    except Exception as e:  
        print(f"Skipped {file_name} — Error: {e}")

combined_df = pd.concat(all_data.values(), ignore_index=True)  

pc_dedup = combined_df[['Emplid', 'STD Hrs']].drop_duplicates()

Loaded: LENOX POSNEXEC LENOX_EXEC 2026-01-03.xlsx (5180 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-01-17.xlsx (5177 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-01-24.xlsx (5183 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-01-31.xlsx (5187 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-02-07.xlsx (5188 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-02-14.xlsx (5187 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-02-21.xlsx (5188 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-02-28.xlsx (5199 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-03-07.xlsx (5196 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-03-21.xlsx (5209 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-04-04.xlsx (5140 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-04-18.xlsx (5145 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-05-16.xlsx (5158 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-05-23.xlsx (5147 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-06-06.xlsx (5153 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-06-13.xlsx (5155 rows)
Loaded: 

In [5]:
anu_dept_ids = [15612095, 15624050, 15625045, 15625045, 15628030, 15632045, 15633025, 15635545, 15635550, 15638045, 15638045, 15652045, 15652065]

ppe_ldr_acps = ppe_ldr_combined[ppe_ldr_combined['gl_deptid'].isin(anu_dept_ids)]

jobtypes = pd.read_excel('C:/Users/kbixby/OneDrive - Northwell Health/Scripts/fte/acp_service_jobtypes.xlsx')
earncodes = pd.read_excel('C:/Users/kbixby/OneDrive - Northwell Health/Scripts/fte/earn_codes.xlsx')

acps_typed = pd.merge(ppe_ldr_acps, jobtypes, how='left', on='jobcode')
acps_earncodes = pd.merge(acps_typed, earncodes, how='left', on='earn_code')
acps_std_hours = pd.merge(acps_earncodes, pc_dedup, how='inner', left_on='emplid', right_on='Emplid')


acps_hours = acps_std_hours[['gl_deptid', 'description', 'jobcode', 'description_1', 'name', 'emplid', 'position', 'pay_end_dt',
 'earn_code', 'ot_hours', 'total_hours', 'pay_type', 'bu', 'type', 'include', 'STD Hrs']]

# 1 is moonlight non flex, 2 is moonlight for flex, 0 is anything else
acps_include = acps_hours[  
    (acps_hours['include'] == 1) | (acps_hours['pay_end_dt'] >= '2026-03-15')  
]  
acps_include.loc[acps_include['STD Hrs'] == 1, 'STD Hrs'] = 37.5

acps_include.loc[acps_include['pay_end_dt'] > '2026-03-15', 'total_hours'] = 0
acps_include.loc[acps_include['pay_end_dt'] <= '2026-03-15', 'ot_hours'] = 0

acps_include['total_ftes'] = acps_include['total_hours'] / (acps_include['STD Hrs'] * 52 /24)

acps_include['ot_ftes'] = acps_include['ot_hours'] / (acps_include['STD Hrs'] * 2)

acps_grouped = acps_include.groupby(['gl_deptid', 'description', 'type', 'pay_end_dt', 'STD Hrs', ])[['total_hours', 'ot_hours','ot_ftes', 'total_ftes']].sum().reset_index()

acps_grouped['reg_ftes'] = acps_grouped['total_ftes'] - acps_grouped['ot_ftes']
display(acps_grouped)
#acps_grouped.to_excel('acp_moonlight.xlsx')

,gl_deptid,description,type,pay_end_dt,STD Hrs,total_hours,ot_hours,ot_ftes,total_ftes,reg_ftes
0,15612095.0,NRSG - Neotat Nurse Practition,Management,2026-03-15,37.5,114.75,0.00,0.00,1.412308,1.412308
1,15612095.0,NRSG - Neotat Nurse Practition,Management,2026-03-21,37.5,0.00,0.00,0.00,0.000000,0.000000
2,15612095.0,NRSG - Neotat Nurse Practition,Management,2026-04-04,37.5,0.00,0.00,0.00,0.000000,0.000000
3,15612095.0,NRSG - Neotat Nurse Practition,Management,2026-04-18,37.5,0.00,14.25,0.19,0.000000,-0.190000
4,15612095.0,NRSG - Neotat Nurse Practition,Management,2026-05-02,37.5,0.00,0.00,0.00,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
1096,15652065.0,ACP Services - Cardio EPS,Senior PA,2026-07-25,37.5,0.00,0.00,0.00,0.000000,0.000000
1097,15652065.0,ACP Services - Cardio EPS,Senior PA,2026-08-08,37.5,0.00,0.00,0.00,0.000000,0.000000
1098,15652065.0,ACP Services - Cardio EPS,Senior PA,2026-08-22,37.5,0.00,0.00,0.00,0.000000,0.000000
1099,15652065.0,ACP Services - Cardio EPS,Senior PA,2026-09-05,37.5,0.00,0.00,0.00,0.000000,0.000000
